# Objectif du notebook 

Date de création : 13/01/2026

Estimation des temps d'arrivée des signaux sur la voie hydro des OBS ("H"). 

In [1]:
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from datetime import datetime


In [2]:
project_root = os.path.abspath(
    os.path.join(os.path.dirname(os.getcwd()), "..", "..", "..")
)
root_groix_data = os.path.join(project_root, "data", "fiberscope_groix_oct_2025")
root_groix_wav = os.path.join(root_groix_data, "wav")
root_groix_metadata = os.path.join(root_groix_data, "metadata")

root_folder = os.path.join(project_root, "real_data_analysis", "fiberscope_groix")
data_folder = os.path.join(root_folder, "data")
img_folder = os.path.join(root_folder, "img")

In [3]:
sys.path.append(project_root)
from publication.publication_figure import PubFigure, color
from real_data_analysis.fiberscope_groix.src.data_processing.process_utils import *

# Chargement des données utiles 

In [4]:
ds_gps = xr.open_dataset(os.path.join(data_folder, "gps.nc"))

# Exemple avec un unique signal

## Chargement des informations relatives aux émissions 

In [5]:
fpath = os.path.join(data_folder, "source_emissions_processed_dtypes.csv")
df_dtypes= pd.read_csv(fpath, index_col=0).to_dict()["dtype"]

fpath = os.path.join(data_folder, "source_emissions_processed.csv")
col_to_parse_date = [key for key in df_dtypes.keys() if "datetime" in df_dtypes[key]]
# Remove these columns from df_dtypes as they will be parsed as dates
for col in col_to_parse_date:
    df_dtypes.pop(col)
# Load dataframe with specified dtypes and parse dates
df = pd.read_csv(fpath, dtype=df_dtypes, parse_dates=col_to_parse_date)

In [6]:
df.dtypes       # Here can verifiy that the dtypes are correctly assigned

Emission datetime              datetime64[ns]
Sequence_id                            object
Point                                  object
Source                                 object
Longueur filée (m)                     object
Hydro distance (m)                    float32
Signal                                 object
Frequency min (Hz)                    float32
Frequency max (Hz)                    float32
Duration (s)                          float32
Nrepeat                                 int32
Trepeat (s)                           float32
Vc carte (V)                          float32
Gain ampli                            float32
File                                   object
File start datetime            datetime64[ns]
Start sample                           object
Emission latitude GPS                 float64
Emission longitude GPS                float64
Emission E GPS                        float64
Emission N GPS                        float64
Emission U GPS                    

### Remarque : calcul de la position de la source 
Le dataframe df contient la position de l'antenne GPS estimée à l'instant de l'émission. En pratique, la source n'est pas colocalisée avec l'antenne et il faut théoriquement prendre en compte ce bras de levier.

Au moins dans le cas des émissions en statique on peut considérer, au regard de l'incertitude sur le positionnement de l'antenne GPS à l'instant émission (incertitude GPS standard + interpolation linéaire entre deux points GPS), que la source est colocalisée avec l'antenne GPS. 

Dans le cas dynamique la longueur filée est importante $\approx$ 15/20 m. Dans ce cas, il peut être plus difficile de négliger le bras de levier entre l'antenne GPS et la source immergée. La longueur filée est connue ainsi que l'immersion de la source (capteur de pression sur la source), on peut ainsi calculer la distance (à la surface) de l'antenne GPS à la source (dans l'axe du navire). Néanmoins, afin de transformer ce bras de levier dans le repère du navire en un offset sur la position dans le repère ENU il est nécessaire de connaitre le cap du navire dans le repère ENU. 

Pour ce faire, on peut estimer le cap du navire à partir de l'estimation du vecteur vitesse du navire à l'instant d'émission : 

$$ V_{gps}^{(ENU)} = [V_e, V_n]^T$$

$$V_e = \frac{E_{gps}(t_{n+1}) - E_{gps}(t_n)}{\Delta t}$$
et 
$$V_n = \frac{N_{gps}(t_{n+1}) - N_{gps}(t_n)}{\Delta t}$$

où $t_n$ et $t_{n+1}$ sont les instants précèdent et successif à l'instant d'émission dans la série temporelle des positions GPS. 

Le cap du navire dans le repère ENU est alors donné par : 

$$\alpha = \arctan{\frac{V_e}{V_n}}$$

La position dans le repère du navire est (hypothèse source dans l'axe du navire):

$$X_s^{(Navire)} = [0, -bdl]^T$$

où $bdl$ est la distance, selon l'axe $y_{navire}$, de l'origine du repère du navire (position de l'antenne GPS) au projeté orthogonal de la position de la source sur la surface. 

Finalement :

$$X_s^{(ENU)} = X_{GPS}^{(ENU)} + R_{\text{Nav to ENU}} X_s^{(Navire)}$$

avec 
$$ R_{\text{Nav to ENU}} = \begin{bmatrix} \cos{\alpha} & \sin{\alpha} \\ -\sin{\alpha} & \cos{\alpha} \end{bmatrix}$$

Proche des OBS la correction pourrait avoir un impacte significatif. 

In [7]:
pfig = PubFigure(
    label_fontsize=18, legend_fontsize=10, ticks_fontsize=16, title_fontsize=20
)

## Estimation du décalage des deux bases de temps UTC 

Le temps UTC de l'hydro source, utilisé pour pointer les temps d'émission, n'est pas parfaitement synchronisé avec le temps UTC GPS (celui des OBS). L'objectif est d'exploiter les émissions au-dessus de chacun des OBSs afin d'estimer ce décalage. 

Cette étape préalable est nécessaire pour la suite de l'estimation des temps d'arrivée. En effet certaine des émissions ne sont pas détectées, dans ce cas, il faut associer les arrivées éparses détectées aux émissions correspondantes. L'alignement des deux bases de temps est nécessaire à cette étape permettant de renforcer la robustesse de la méthode. 

### Détails

* Les temps d'arrivées théoriques sont donnés en temps UTC de l'hydrophone : $t_{th_{arr}}^{(Hydro)}$
* Les temps d'arrivées mesurés sont donnés en temps UTC de l'OBS : $t_{arr}^{(OBS)}$

Le shift entre les deux bases de temps est donné (aux erreurs de mesures et de modélisation près) par : 
$$ \tau_{Hydro} = t_{th_{arr}}^{(Hydro)} - t_{arr}^{(OBS)}$$

Ici le shift est évalué sur la différence de temps de propagation : 

* Le temps de propagation théorique est donné par : 
$$\tau_{th} = t_{th_{arr}}^{(Hydro)} - t_{emission}^{(Hydro)}$$ 
* Le temps de propagation mesuré est donné par :
$$\tau_{mes} = t_{arr}^{(OBS)} - t_{emission}^{(Hydro)}$$

et on a donc : 

$$\tau_{Hydro} = \tau_{th} - \tau_{mes} $$

### Sélection d'une partie des émissions

In [8]:
subset_params = {
    "Signal": "chirp",  # "chirp" or "sinus"
    "Source": "fixed",  # "fixed" or "trailed"
    "Nrepeat": 10,  # number of repeats
    "Vc carte (V)": None,  # Source amplitude (V)
    "Emission datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
}

df_sel = select_dataframe_subset(df, subset_params)

In [9]:
df_sel

,Emission datetime,Sequence_id,Point,Source,Longueur filée (m),Hydro distance (m),Signal,Frequency min (Hz),Frequency max (Hz),Duration (s),...,Emission longitude AIS,Emission E AIS,Emission N AIS,Emission U AIS,Emission interpolated E GPS,Emission interpolated N GPS,Emission interpolated U GPS,Emission interpolated E AIS,Emission interpolated N AIS,Emission interpolated U AIS
0,2025-10-15 07:47:33.449765625,12,1,fixed,2,1.30,chirp,80.0,1000.0,5.0,...,-3.471958,-459.518339,-756.016082,30.422226,-453.658226,-769.759689,30.420998,-459.211282,-755.983503,30.422252
1,2025-10-15 07:47:43.449765625,12,1,fixed,2,1.30,chirp,80.0,1000.0,5.0,...,-3.471946,-458.628258,-755.921645,30.422301,-454.762126,-769.490064,30.454415,-458.321203,-755.889071,30.455789
2,2025-10-15 07:47:53.449765625,12,1,fixed,2,1.30,chirp,80.0,1000.0,5.0,...,-3.471934,-457.738184,-755.827220,30.519376,-455.866028,-769.220443,30.517906,-457.431127,-755.794642,30.519402
3,2025-10-15 07:48:03.449765625,12,1,fixed,2,1.30,chirp,80.0,1000.0,5.0,...,-3.471922,-456.848103,-755.732784,30.519451,-456.969926,-768.950814,30.517860,-456.541045,-755.700206,30.519476
4,2025-10-15 07:48:13.449765625,12,1,fixed,2,1.30,chirp,80.0,1000.0,5.0,...,-3.471910,-455.958021,-755.638347,30.519525,-458.073822,-768.681181,30.484351,-455.650962,-755.605765,30.486088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,2025-10-16 14:33:39.478031250,167,1S,fixed,8,1.35,chirp,80.0,1000.0,5.0,...,-3.471910,-455.883408,-741.561236,30.811185,-468.122812,-744.620133,30.809943,-455.894962,-741.546879,30.811186
646,2025-10-16 14:33:49.478031250,167,1S,fixed,8,1.35,chirp,80.0,1000.0,5.0,...,-3.471906,-455.662048,-741.836290,30.811169,-467.914170,-744.630441,30.809957,-455.673602,-741.821933,30.811170
647,2025-10-16 14:33:59.478031250,167,1S,fixed,8,1.35,chirp,80.0,1000.0,5.0,...,-3.471904,-455.440688,-742.111343,30.811153,-467.705528,-744.640748,30.809971,-455.452242,-742.096986,30.811153
648,2025-10-16 14:34:09.478031250,167,1S,fixed,8,1.35,chirp,80.0,1000.0,5.0,...,-3.471900,-455.219328,-742.386396,30.811136,-467.496885,-744.651056,30.809985,-455.230882,-742.372039,30.811137


In [ ]:
obs_id = 1  # Correspondance OBS 1 = OBS 4, OBS 2 = OBS 5, OBS 3 = OBS 6

# plot = True
# savefig = True
plot = False
savefig = False
verbose = False
plot_zoom = False

# Define image folder for preprocessing plots
img_preprocess_folder = os.path.join(
    img_folder, "reception", "arrivals_detection", "preprocessing"
)

# Convention Gen_Axes_D_V4 (Cf ELOBSBin2Wav.py)
channels_order = {
    "Z": 0,
    "X": 1,
    "Y": 2,
    "H": 3,
}
used_channel = "H"

# TODO : check to remove this or to moove it earlier
t_hydro_source_offset = 27    # seconds

# Window parameters
pre_reception_time = 5.0    # seconds before reception to include in the window
post_reception_time = 10.0   # seconds after reception to include in the window

# Correct window for hydrophone to source offset
pre_reception_time -= t_hydro_source_offset
post_reception_time += t_hydro_source_offset

origin_keys = df_sel.columns.to_list()
processed_keys = origin_keys
processed_data = {key: [] for key in processed_keys}

# List available wav files for each OBS (to avoid reloading them for each sequence)
wav_start_times_dict = {}
start_datetime_arr_dict = {}
for obs_id in [1, 2, 3]:
    wav_start_times, start_datetime_arr = get_available_wav_files(obs_id)
    wav_start_times_dict[obs_id] = wav_start_times
    start_datetime_arr_dict[obs_id] = start_datetime_arr

# # # VC = 4V
# # # seq_OBS1 = 151  # Emissions on OBS1 (2 m)
# # seq_OBS1 = 155  # Emissions on OBS1 (5 m)
# # seq_OBS2 = 116  # Emissions on OBS2
# # seq_OBS3 = 131  # Emissions on OBS3

# # seq_OBS1 = 167  # Emissions on OBS1 (8 m)
# seq_OBS1 = 24  # Emissions on OBS1 (5 m)
# seq_OBS2 = 116  # Emissions on OBS2
# seq_OBS3 = 131  # Emissions on OBS3

# sel_seq_id = [90]  # Séquence correstement détectées pour VC = 4V (inspection visuelle)

sel_seq_id_4v = [12, 24, 35, 39, 51, 80, 90, 116, 127, 131, 143, 151 ,155, 167]       # Séquence correstement détectées pour VC = 4V (inspection visuelle)
# sel_sequence_id = [str(seq_id) for seq_id in sel_seq_id_4v]
# print(f"Selected sequences: OBS1, OBS2, OBS3 -> {sel_sequence_id}")

# # # VC = 2V
sel_seq_id_2v = [13, 25, 40, 81, 117, 132, 156]
# sel_sequence_id = [str(seq_id) for seq_id in sel_seq_id_2v]

# # seq_OBS1 = 156  # Emissions on OBS1 (5 m)
# # seq_OBS2 = 117  # Emissions on OBS2
# # seq_OBS3 = 132  # Emissions on OBS3
# sel_sequence_id = [str(seq_OBS1), str(seq_OBS2), str(seq_OBS3)]
# print(f"Selected sequences: OBS1, OBS2, OBS3 -> {sel_sequence_id}")

# VC = 1V
sel_seq_id_1v = [14, 26, 41, 118, 133, 157]

sel_sequence_id = [
    str(seq_id) for seq_id in sel_seq_id_2v + sel_seq_id_4v + sel_seq_id_1v
]
# sel_sequence_id = df_sel["Sequence_id"].unique()
print(f"Sequences 4V : {len(sel_seq_id_4v)}")
print(f"Sequences 2V : {len(sel_seq_id_2v)}")
print(f"Sequences 1V : {len(sel_seq_id_1v)}")
print(f"Total selected sequences: {len(sel_sequence_id)} -> {sel_sequence_id}")

for seq_id in sel_sequence_id:
    df_sequence = df_sel.loc[df_sel["Sequence_id"] == seq_id]

    # # Correct offset # TODO remove
    # # print(df_sequence["Emission datetime"])
    # df_sequence["Emission datetime"] = df_sequence["Emission datetime"] + pd.Timedelta(t_hydro_source_offset, "s")
    # print(df_sequence["Emission datetime"])

    new_data = {}
    # Lood over receivers
    for obs_id in [1, 2, 3]:

        # Get available wav files for the selected OBS
        wav_start_times = wav_start_times_dict[obs_id]
        start_datetime_arr = start_datetime_arr_dict[obs_id]

        ### Load signal of interest ###
        # Get all theoretical arrival time
        emissions_datetime = []
        th_arrivals_datetime = []
        th_propagation_delay = []
        th_arrivals_seconds_from_start = []
        for emission_id in range(df_sequence.shape[0]):
            emission_i = df_sequence.iloc[emission_id]
            emission_i_datetime = emission_i["Emission datetime"].to_pydatetime(warn=False)
            emissions_datetime.append(emission_i_datetime)

            if emission_id == 0:        # First emission in sequence 
                # Find the corresponding wav file (for now we assume that the sequence fits in a single wav file)
                wav_fpath, wav_start_datetime = get_wav_file_for_emission(
                    emission_datetime=emission_i_datetime,
                    start_datetime_arr=start_datetime_arr,
                    wav_start_times=wav_start_times,
                )
                # print(f"Corresponding wav file (OBS{obs_id}): {wav_fpath}")

            # Emission position
            emission_i_pos = [
                emission_i["Emission interpolated E GPS"],
                emission_i["Emission interpolated N GPS"],
                emission_i["Emission interpolated U GPS"],
            ]
            # Theoretical time of arrival
            emission_i_reception_datetime, tr_i, prop_time_i = get_tr_apriori(
                emission_i_pos,
                wav_start_datetime,
                emission_i_datetime,
                obs_id,
            )
            th_propagation_delay.append(prop_time_i)
            th_arrivals_datetime.append(emission_i_reception_datetime)
            th_arrivals_seconds_from_start.append(tr_i)

        # Convert to numpy arrays
        th_propagation_delay = np.array(th_propagation_delay)
        th_arrivals_datetime = np.array(th_arrivals_datetime)
        th_arrivals_seconds_from_start = np.array(th_arrivals_seconds_from_start)

        # First emission in the sequence
        first_emission_in_sequence_datetime = emissions_datetime[0] 
        first_emission_reception_datetime = th_arrivals_datetime[0]
        tr_first = th_arrivals_seconds_from_start[0]
        # Last emission in sequence
        last_emission_in_sequence_datetime = emissions_datetime[-1]
        last_emission_reception_datetime = th_arrivals_datetime[-1]
        tr_last = th_arrivals_seconds_from_start[-1]
        # print(
        #     f"Processing sequence {seq_id} from {first_emission_in_sequence_datetime} to {last_emission_in_sequence_datetime}"
        # )

        # Load the wav file
        # print("Loading wav file...")
        # Read the wav file
        signal, fs = sf.read(wav_fpath)
        # Select the channel
        signal = signal[:, channels_order[used_channel]]
        # Center the signal
        signal = signal - np.mean(signal)

        # Compute source position considering the offset for the current emission   (Source, Longueur filée)
        # TODO : implement position correction if needed

        # Select the time window of interest for current sequence (all emissions in the sequence + pre/post times)
        t_start_win = tr_first - pre_reception_time
        t_end_win = tr_last + post_reception_time

        # Convert in samples
        n_samp_start_win = int(t_start_win * fs)
        n_samp_end_win = int(t_end_win * fs)
        # Slice signal
        signal_win = signal[n_samp_start_win:n_samp_end_win]
        wav_end_datetime = wav_start_datetime + pd.Timedelta(signal.shape[0] * 1/fs, "s")
        t_dt = pd.date_range(wav_start_datetime, wav_end_datetime, freq=f"{1/fs}s", inclusive="left")
        t_win = t_dt[n_samp_start_win:n_samp_end_win]
        # print(f"Original signal from {wav_start_datetime} to {wav_end_datetime}")
        # print(f"Selected window from {t_win[0]} to {t_win[-1]}")

        ### Get arrivals ###
        peaks_idx, peak_times, t_arrivals, sig_mf, signal_params, signal_win_filter = (
            get_arrivals(
                signal_win,
                t_win,
                df_sequence,
                fs,
                verbose=verbose,
            )
        )
        signal_win = signal_win_filter

        ### Plot arrivals ###
        if plot:
            sequence_info = {
                "seq_id": seq_id,
                "obs_id": obs_id,
                "vc_carte": df_sequence["Vc carte (V)"].iloc[0],
                "signal_type": df_sequence["Signal"].iloc[0],
                "emission_type": df_sequence["Source"].iloc[0],
            }
            nperseg=64
            noverlap=int(nperseg*0.5)

            plot_arrivals_detection(
                t_win=t_win,
                signal_win=signal_win,
                sig_mf=sig_mf,
                t_arrivals=t_arrivals,
                peaks_idx=peaks_idx,
                peak_times=peak_times,
                sequence_info=sequence_info,
                signal_params=signal_params,
                plot_last_first=False,
                first_emission_reception_datetime=first_emission_reception_datetime,
                last_emission_reception_datetime=last_emission_reception_datetime,
                t_hydro_source_offset=t_hydro_source_offset,
                save=savefig,
                img_root=img_preprocess_folder,
                fs=fs,
                nperseg=nperseg,
                noverlap=noverlap,
                first_emission_in_sequence_datetime=None,
                last_emission_in_sequence_datetime=None,
                verbose=verbose,
                plot_zoom=plot_zoom,
            )

            plt.close("all")

        ### Compute quality metrics for the detected arrivals ###
        # Derive peak signal to noise ratio (PSNR) on matched filtered signal
        psnr_arrivals = detected_arrivals_psnr(
            sig_mf, peaks_idx, signal_params, fs, plot=False
        )
        # print(psnr_arrivals)

        # Convert t_arrivals into datetime.datetime
        t_arrivals_dt = np.array([t_arr.to_pydatetime(warn=False) for t_arr in t_arrivals])

        valid_detection = np.zeros_like(emissions_datetime, dtype=bool)

        if len(t_arrivals_dt) < len(emissions_datetime):
            print(
                f"Warning: only {len(t_arrivals_dt)} arrivals detected for {len(emissions_datetime)} emissions in sequence {seq_id} OBS{obs_id}"
            ) 
            # Pad in case not all peaks are detected
            psnr_arrivals_full = np.full_like(emissions_datetime, np.nan, dtype=float)
            t_arrivals_full = np.full_like(emissions_datetime, pd.NaT)
            t_arrivals_dt_full = np.full_like(emissions_datetime, pd.NaT)

            # Associate arrivals to closest theoretical arrival
            th_arrivals_datetime_copy = th_arrivals_datetime.copy()
            for i_t_arr, t_arr_dt in enumerate(t_arrivals_dt):
                # Find closest
                closest_th_arr_idx = np.argmin(np.abs(th_arrivals_datetime_copy - t_arr_dt))
                # Remove this theoretical arrival from the copy to avoid double matching
                th_arrivals_datetime_copy = np.delete(
                    th_arrivals_datetime_copy, closest_th_arr_idx
                )
                # Replace in padded arrays
                t_arrivals_dt_full[closest_th_arr_idx] = t_arr_dt
                t_arrivals_full[closest_th_arr_idx] = t_arrivals[i_t_arr]
                psnr_arrivals_full[closest_th_arr_idx] = psnr_arrivals[i_t_arr]

                # Set valid_detection flag to true
                valid_detection[closest_th_arr_idx] = True

        else:
            t_arrivals_full = t_arrivals
            t_arrivals_dt_full = t_arrivals_dt
            psnr_arrivals_full = psnr_arrivals
            valid_detection[:] = True

        # Derive propagation time
        try:
            meas_propagation_delay = t_arrivals_dt_full - np.array(emissions_datetime)
            meas_propagation_delay = np.array([t.total_seconds() for t in meas_propagation_delay])
        except:
            print("flag")

        # Add new data for current obs
        new_data[f"Arrival datetime OBS{obs_id}"] = list(t_arrivals_full)
        new_data[f"Theoretical propagation time OBS{obs_id}"] = list(th_propagation_delay)
        new_data[f"Measured propagation time OBS{obs_id}"] = list(meas_propagation_delay)
        new_data[f"PSNR OBS{obs_id}"] = list(psnr_arrivals_full)
        new_data[f"Valid detection OBS{obs_id}"] = list(valid_detection)

    # Copy data for processed emissions
    for key in origin_keys:
        processed_data[key].extend(df_sequence[key].values)
    for key in new_data:
        if key in processed_data.keys():
            processed_data[key].extend(new_data[key])
        else:
            processed_data[key] = new_data[key]

    # processed_data[""]

# for key in processed_data.keys():
#     print(len(processed_data[key]))
#     if len(processed_data[key]) != 150:
#         print(key)
# Convert to dataframe
df_processed = pd.DataFrame(processed_data)

In [ ]:
df_processed = df_processed.loc[
    df_processed["Valid detection OBS1"]
    & df_processed["Valid detection OBS2"]
    & df_processed["Valid detection OBS3"]
]

In [ ]:
# Estimation du décalage temporel entre les deux bases de temps
time_diff = {}
sel_sequence_id = df_processed["Sequence_id"].unique()
for i, seq_id in enumerate(sel_sequence_id):
    df_sequence = df_processed.loc[df_processed["Sequence_id"] == seq_id]
    time_diff[seq_id] = {}
    # Store emission datetime
    time_diff[seq_id]["first_emission_datetime"] = df_sequence["Emission datetime"].iloc[0].to_pydatetime(warn=False)

    print(f"Sequence {seq_id} - Emission on OBS{i%3+1}") 

    for obs_id in [1, 2, 3]:

        # Store first position
        time_diff[seq_id][f"Theoretical propagation time OBS{obs_id}"] = df_sequence[
            f"Theoretical propagation time OBS{obs_id}"
        ].iloc[0]

        print(f"OBS{obs_id}:")
        # print(f'\tEmissions: {df_sequence["Emission datetime"]}')
        print(
            f'\tTheoretical propagation time (mean) : {np.mean(df_sequence[f"Theoretical propagation time OBS{obs_id}"])} s'
        )
        print(f'\tPropagation dist (mean th propa time): {np.mean(df_sequence[f"Theoretical propagation time OBS{obs_id}"]) * 1500} m')
        print(
            f'\tMeasured propagation time (mean) : {np.mean(df_sequence[f"Measured propagation time OBS{obs_id}"])} s'
        )
        print(
            f'\tPropagation dist (mean meas propa time): {np.mean(df_sequence[f"Measured propagation time OBS{obs_id}"]) * 1500} m'
        )

        time_diffs = (
            df_sequence[f"Theoretical propagation time OBS{obs_id}"]
             - df_sequence[f"Measured propagation time OBS{obs_id}"]
        )
        # print(time_diffs)
        time_offset_median = np.nanmedian(time_diffs)
        time_offset_mean = np.nanmean(time_diffs)
        time_offset_std = np.nanstd(time_diffs)
        # print(f"Estimated time offset for OBS{obs_id}: {time_offset} s")
        print(
            f"\tEstimated time offset: \n\t\tmedian={time_offset_median:.3f} s \n\t\tmean={time_offset_mean:.3f} s \n\t\tstd={time_offset_std:.3f} s"
        )

        # Store
        time_diff[seq_id][f"OBS{obs_id}"] = {
            "median": time_offset_median,
            "mean": time_offset_mean,
            "std": time_offset_std,
        }


print(time_diff)